# Silver Layer — Customers Transformation

This notebook transforms data from the Bronze layer and writes it to the Silver layer.

It also runs data quality checks before writing to the Silver table. The data quality check functions are defined in a separate notebook and imported here to use before writing.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### Transformations
Reading from `bronze.customers`, applying cleaning steps, and storing the result in `silver_customers_df`.

In [0]:
silver_customers_df = spark.read.table(f'{catalog_name}.{bronze_schema}.customers')\
                                .filter(col('customer_id').isNotNull())\
                                .dropDuplicates(['customer_id'])\
                                .withColumn('customer_city',lower(trim(col('customer_city'))))\
                                .withColumn('customer_state',lower(trim(col('customer_state'))))\
                                .withColumn('silver_processed_at',current_timestamp())


### Data Quality Checks
Running the three reusable DQ checks on the transformed DataFrame before writing to Silver.


In [0]:
check_not_null(silver_customers_df,['customer_id','customer_unique_id'])
check_unique(silver_customers_df,['customer_id'])
check_row_count(silver_customers_df,f'{catalog_name}.{bronze_schema}.customers',70,100)

check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED


### Write to Silver
Writing the validated DataFrame to the Silver Delta table.

In [0]:
silver_customers_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.customers')

In [0]:
display(spark.read.table(f'{catalog_name}.{silver_schema}.customers').count())

99441